# NER Project Pipeline - CoNLL-2003 + MasakhaNER
This notebook implements the complete NER pipeline specification as outlined in the TODO.

## 1. Environment & Data Acquisition
Install dependencies and load datasets.

In [1]:
import torch # Add this line first!
from transformers import AutoTokenizer

model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)


In [2]:
%pip install datasets==2.14.0 transformers seqeval torch accelerate evaluate
from datasets import load_dataset, concatenate_datasets, DatasetDict

print("Loading CoNLL-2003...")
conll = load_dataset("conll2003")

masakha_langs = ['amh', 'hau', 'ibo', 'kin', 'lug', 'luo', 'pcm', 'swa', 'wol', 'yor']
masakha_datasets = {}
for lang in masakha_langs:
    print(f"Loading MasakhaNER {lang}...")
    masakha_datasets[lang] = load_dataset("masakhane/masakhaner", lang)

print("Loading supplemental Amharic...")
supp_amharic = load_dataset("rasyosef/amharic-named-entity-recognition")

print(f"CoNLL train rows: {len(conll['train'])}")
for lang in masakha_langs:
    print(f"MasakhaNER {lang} train rows: {len(masakha_datasets[lang]['train'])}")
print(f"Supplemental Amharic train rows: {len(supp_amharic['train'])}")

Note: you may need to restart the kernel to use updated packages.
Loading CoNLL-2003...
Loading MasakhaNER amh...


Loading MasakhaNER hau...
Loading MasakhaNER ibo...
Loading MasakhaNER kin...
Loading MasakhaNER lug...
Loading MasakhaNER luo...
Loading MasakhaNER pcm...
Loading MasakhaNER swa...
Loading MasakhaNER wol...
Loading MasakhaNER yor...
Loading supplemental Amharic...
CoNLL train rows: 14041
MasakhaNER amh train rows: 1750
MasakhaNER hau train rows: 1912
MasakhaNER ibo train rows: 2235
MasakhaNER kin train rows: 2116
MasakhaNER lug train rows: 1428
MasakhaNER luo train rows: 644
MasakhaNER pcm train rows: 2124
MasakhaNER swa train rows: 2109
MasakhaNER wol train rows: 1871
MasakhaNER yor train rows: 2171
Supplemental Amharic train rows: 3465


## 2. Explore & Profile the Data
Check tag schemas and distributions.

In [3]:
# Inspect CoNLL tags (has MISC instead of DATE)
print("CoNLL tags:", conll["train"].features["ner_tags"])
# Inspect MasakhaNER tags (has DATE instead of MISC)
print("MasakhaNER tags (Amh):", masakha_datasets["amh"]["train"].features["ner_tags"])
# Inspect Supplemental Amharic tags (column is 'ner_tags', not 'tags')
print("Supp Amharic tags:", supp_amharic["train"].features["ner_tags"])
print("Supp Amharic columns:", supp_amharic["train"].column_names)

# ─── Schema summary ───────────────────────────────────────────────────────
# CoNLL:      0=O 1=B-PER 2=I-PER 3=B-ORG 4=I-ORG 5=B-LOC 6=I-LOC 7=B-MISC 8=I-MISC
# MasakhaNER: 0=O 1=B-PER 2=I-PER 3=B-ORG 4=I-ORG 5=B-LOC 6=I-LOC 7=B-DATE 8=I-DATE
# rasyosef:   0=O 1=B-PER 2=I-PER 3=B-ORG 4=I-ORG 5=B-LOC 6=I-LOC 7=B-TIME 8=I-TIME 9=B-TTL 10=I-TTL
#
# Unification plan:
# - CoNLL MISC (7,8)        -> O (0) : not in our PER/ORG/LOC taxonomy
# - MasakhaNER DATE (7,8)   -> O (0) : same
# - rasyosef TIME (7,8)     -> O (0) : same
# - rasyosef TTL (9,10)     -> O (0) : title tag not in taxonomy
# After remapping, all datasets share: 0=O 1=B-PER 2=I-PER 3=B-ORG 4=I-ORG 5=B-LOC 6=I-LOC

CoNLL tags: Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None), length=-1, id=None)
MasakhaNER tags (Amh): Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE'], id=None), length=-1, id=None)
Supp Amharic tags: Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-TIME', 'I-TIME', 'B-TTL', 'I-TTL'], id=None), length=-1, id=None)
Supp Amharic columns: ['tokens', 'ner_tags']


## 3. Deduplicate & Merge Amharic Data
Merge supplemental Amharic data into MasakhaNER Amharic train split, preventing leakage.

In [4]:
import hashlib
from datasets import ClassLabel, Sequence

# Build leakage-prevention hash set from val/test
amh_val_test_texts = set()
for split in ['validation', 'test']:
    for row in masakha_datasets['amh'][split]:
        amh_val_test_texts.add(hashlib.md5(' '.join(row['tokens']).encode('utf-8')).hexdigest())

print(f"Val/test hashes built: {len(amh_val_test_texts)} unique sentences")

def is_not_leakage(row):
    h = hashlib.md5(' '.join(row['tokens']).encode('utf-8')).hexdigest()
    return h not in amh_val_test_texts

filtered_supp = supp_amharic['train'].filter(is_not_leakage)
print(f"Supplemental rows after dedup: {len(filtered_supp)} / {len(supp_amharic['train'])} kept")

# Collapse TIME (7,8) and TTL (9,10) → O (0)
def remap_supp_tags(row):
    row['ner_tags'] = [0 if t >= 7 else t for t in row['ner_tags']]
    return row

filtered_supp = filtered_supp.map(remap_supp_tags)

# Cast ClassLabel metadata to match MasakhaNER's schema
masakha_ner_feature = Sequence(ClassLabel(
    names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']
))
filtered_supp = filtered_supp.cast_column('ner_tags', masakha_ner_feature)

# Add synthetic id column (rasyosef has none)
filtered_supp = filtered_supp.map(
    lambda row, idx: {'id': str(idx)}, with_indices=True
)
filtered_supp = filtered_supp.select_columns(['id', 'tokens', 'ner_tags'])

# Merge into Amharic train only
masakha_datasets['amh']['train'] = concatenate_datasets([
    masakha_datasets['amh']['train'].select_columns(['id', 'tokens', 'ner_tags']),
    filtered_supp
])
print(f"Merged Amharic train: {len(masakha_datasets['amh']['train'])} rows")


Val/test hashes built: 750 unique sentences


Filter:   0%|          | 0/3465 [00:00<?, ? examples/s]

Supplemental rows after dedup: 3465 / 3465 kept


Map:   0%|          | 0/3465 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3465 [00:00<?, ? examples/s]

Map:   0%|          | 0/3465 [00:00<?, ? examples/s]

Merged Amharic train: 5215 rows


c:\Users\My Device\AppData\Local\Programs\Python\Python311\Lib\site-packages\datasets\table.py:1421: FutureWarning: promote has been superseded by mode='default'.
  table = cls._concat_blocks(blocks, axis=0)


## 4. Build the Combined Training Set (10 languages + English)
Add language column and concatenate datasets.

In [5]:
from datasets import ClassLabel, Sequence

def add_lang_column(dataset, lang_code):
    return dataset.map(lambda x: {"language": lang_code})

conll = add_lang_column(conll, "eng")
for lang in masakha_langs:
    masakha_datasets[lang] = add_lang_column(masakha_datasets[lang], lang)

# Remap extra tags to O (integer values only)
def unify_tags(example):
    example["ner_tags"] = [0 if tag >= 7 else tag for tag in example["ner_tags"]]
    return example

conll = conll.map(unify_tags)
for lang in masakha_langs:
    masakha_datasets[lang] = masakha_datasets[lang].map(unify_tags)

# Cast ClassLabel metadata to the shared 7-class schema
unified_feature = Sequence(ClassLabel(
    names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
))
conll = conll.cast_column('ner_tags', unified_feature)
for lang in masakha_langs:
    masakha_datasets[lang] = masakha_datasets[lang].cast_column('ner_tags', unified_feature)

# Select only the columns we need
columns_to_keep = ["id", "tokens", "ner_tags", "language"]
conll = conll.select_columns(columns_to_keep)
for lang in masakha_langs:
    masakha_datasets[lang] = masakha_datasets[lang].select_columns(columns_to_keep)

# Concatenate all languages
train_sets = [conll["train"]] + [masakha_datasets[l]["train"] for l in masakha_langs]
val_sets   = [conll["validation"]] + [masakha_datasets[l]["validation"] for l in masakha_langs]
test_sets  = [conll["test"]] + [masakha_datasets[l]["test"] for l in masakha_langs]

combined_dataset = DatasetDict({
    "train":      concatenate_datasets(train_sets).shuffle(seed=42),
    "validation": concatenate_datasets(val_sets),
    "test":       concatenate_datasets(test_sets),
})

print("Combined Train Size:", len(combined_dataset["train"]))
print("Combined Val Size:  ", len(combined_dataset["validation"]))
print("Combined Test Size: ", len(combined_dataset["test"]))


Map:   0%|          | 0/5215 [00:00<?, ? examples/s]

Map:   0%|          | 0/5215 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5215 [00:00<?, ? examples/s]

Combined Train Size: 35866
Combined Val Size:   5868
Combined Test Size:  8729


## 5. Preprocess: Tokenize & Align BIO Labels
Tokenize with `xlm-roberta-base` and align subwords.

In [6]:
from transformers import AutoTokenizer

model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True, 
        max_length=128,
        padding=False
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # First subword gets the real label
            else:
                label_ids.append(-100) # Subsequent subwords ignored (no sub-token supervision)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = combined_dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/35866 [00:00<?, ? examples/s]

## 6. Pick & Configure the Model
Initialize `AutoModelForTokenClassification`.

In [11]:
import torch
from transformers import AutoModelForTokenClassification

MODEL_CHECKPOINT = "xlm-roberta-base"

label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC"
]

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Number of labels:", len(label_list))

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.to(device)

print("Model loaded successfully.")

Device: cpu
Number of labels: 7


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully.


## 7. Train
Set up Trainer and training arguments.

In [12]:
from transformers import (
    TrainingArguments,
    DataCollatorForTokenClassification,
    Trainer
)

OUTPUT_DIR = "./models/ner-multilingual-model"

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Evaluation / checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Select best checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Reproducibility
    seed=42,

    # Logging
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",

    # Mixed precision
    fp16=torch.cuda.is_available(),
)

print("Training configuration ready.")

Training configuration ready.


In [14]:
import evaluate

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    # Convert logits → predicted label IDs
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        pred_sequence = []
        label_sequence = []

        for pred_id, label_id in zip(prediction, label):

            # Ignore special tokens and subword positions
            if label_id == -100:
                continue

            pred_sequence.append(label_list[pred_id])
            label_sequence.append(label_list[label_id])

        true_predictions.append(pred_sequence)
        true_labels.append(label_sequence)

    # Entity-level evaluation using seqeval
    results = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("compute_metrics is ready.")

compute_metrics is ready.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

C:\Users\My Device\AppData\Local\Temp\ipykernel_10184\720657926.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
train_result = trainer.train()

c:\Users\My Device\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


## 8. Evaluate at Entity Level (seqeval)
Compute metrics per-language and globally.

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Uncomment to start training
# trainer.train() 
# trainer.evaluate()


## 9. Error Analysis
Analyze per-language metrics and breakdown.

In [ ]:
# Function to run seqeval per language on the test set
def evaluate_per_language(trainer, test_dataset, lang_code):
    lang_test = test_dataset.filter(lambda x: x["language"] == lang_code)
    print(f"Evaluating {lang_code}...")
    metrics = trainer.evaluate(lang_test)
    print(metrics)
    
# for lang in masakha_langs + ["eng"]:
#     evaluate_per_language(trainer, tokenized_datasets["test"], lang)


## 10. Package & Deploy
Inference pipeline.

In [ ]:
from transformers import pipeline

# ner_pipeline = pipeline("ner", model="./models/ner-multilingual-model/checkpoint-<FINAL>", aggregation_strategy="simple")

# example_text = "Nelson Mandela was born in Mvezo, South Africa."
# entities = ner_pipeline(example_text)
# print(entities)
